In [ ]:
import json
import re
from pathlib import Path
from IPython.display import display, HTML, Markdown
import pandas as pd
from data_cleaning.parse import clean_answer


In [ ]:
# Load analysis
analysis_dir = Path("data_cleaning/data/analysis")
with open(analysis_dir / "summary.json") as f:
    summary = json.load(f)
with open(analysis_dir / "test_cases.json") as f:
    test_cases = json.load(f)

# Load raw files
raw_dir = Path("data_cleaning/data/raw")
all_files = list(raw_dir.rglob("*.md"))
print(f"✅ Loaded {len(all_files)} raw files and analysis of {summary['total_files']} files")


In [ ]:
def compare_document(index: int = 0, doc_id: str = None):
    """Show original vs. cleaned text for a document."""
    if doc_id:
        filepath = next(
            (f for f in all_files if f.stem == doc_id or doc_id in str(f)),
            None,
        )
    else:
        filepath = all_files[index]

    if not filepath:
        print(f"❌ Document not found: {doc_id}")
        return

    with open(filepath, "r", encoding="utf-8") as f:
        content = f.read()

    # Extract body (skip frontmatter)
    if content.lstrip().startswith("---"):
        parts = content.split("---", 2)
        body = parts[2] if len(parts) >= 3 else content
    else:
        body = content

    cleaned = clean_answer(body)

    print(f"### 📄 Document: `{filepath.stem}`")
    print(f"**Course/Section**: {filepath.parent.parent.name}/{filepath.parent.name}")
    display(HTML(f"""
        <div style="display: grid; grid-template-columns: 1fr 1fr; gap: 20px;">
            <div>
                <h3>📝 Original Body</h3>
                <pre style="background: #f8f9fa; padding: 15px; border-radius: 5px; max-height: 400px; overflow: auto;">
{body[:1000]}{'...' if len(body) > 1000 else ''}
                </pre>
            </div>
            <div>
                <h3>✅ Cleaned Body</h3>
                <pre style="background: #e8f5e9; padding: 15px; border-radius: 5px; max-height: 400px; overflow: auto;">
{cleaned[:1000]}{'...' if len(cleaned) > 1000 else ''}
                </pre>
            </div>
        </div>
    """))

# Example usage
compare_document(0)  # First document


In [ ]:
def test_feature(feature: str, example_index: int = 0):
    """Test cleaning on a specific feature."""
    if feature not in summary["feature_stats"]:
        print(f"❌ Feature '{feature}' not found")
        return

    examples = summary["feature_stats"][feature]["examples"]
    if not examples:
        print(f"❌ No examples for '{feature}'")
        return

    ex = examples[example_index]
    filepath = Path(ex["filepath"])

    with open(filepath, "r", encoding="utf-8") as f:
        content = f.read()

    if content.lstrip().startswith("---"):
        parts = content.split("---", 2)
        body = parts[2] if len(parts) >= 3 else content
    else:
        body = content

    # Find the feature in the body
    import re
    pattern = FEATURES.get(feature, r".+")
    matches = re.findall(pattern, body, re.MULTILINE | re.DOTALL)
    matched_text = matches[example_index] if matches else body[:200]

    cleaned = clean_answer(matched_text)

    print(f"### 🧪 Testing '{feature}'")
    print(f"**File**: {ex['filepath']} | **ID**: {ex['id']}")
    display(HTML(f"""
        <div style="display: grid; grid-template-columns: 1fr 1fr; gap: 20px;">
            <div>
                <h4>Input (Feature Example)</h4>
                <pre style="background: #fff3e0; padding: 10px; border-radius: 5px;">
{matched_text[:500]}{'...' if len(matched_text) > 500 else ''}
                </pre>
            </div>
            <div>
                <h4>Output (Cleaned)</h4>
                <pre style="background: #e8f5e9; padding: 10px; border-radius: 5px;">
{cleaned[:500]}{'...' if len(cleaned) > 500 else ''}
                </pre>
            </div>
        </div>
    """))

# Example: Test links
test_feature("link")


In [ ]:
def feature_coverage_report():
    """Show which features are handled by the cleaner."""
    handled_features = {
        "bold": "✅ (removes **)",
        "italic": "✅ (removes *)",
        "strikethrough": "✅ (removes ~~)",
        "inline_code": "✅ (removes `)",
        "code_block_fenced": "✅ (preserves ```)",
        "code_block_indented": "✅ (preserves as ```)",
        "link": "✅ (keeps text)",
        "image": "✅ (removes entirely)",
        "image_placeholder": "✅ (removes <{IMAGE:...}>)",
        "html_tag": "✅ (removes)",
        "html_comment": "✅ (removes)",
        "jinja_block": "✅ (removes)",
        "jinja_var": "✅ (removes)",
        "header": "✅ (removes #)",
        "blockquote": "✅ (removes >)",
        "list_ul": "✅ (removes -/*/+)",
        "list_ol": "✅ (removes 1.)",
        "task_list": "✅ (removes [x])",
        "horizontal_rule": "✅ (removes ---)",
        "table": "✅ (keeps text)",
        "escape": "✅ (removes \)",
        "footnote": "✅ (removes [^1])",
    }

    print("### 📊 Feature Coverage Report")
    print("| Feature | Status | Occurrences | Files |")
    print("|---------|--------|-------------|-------|")
    for feature, data in sorted(
        summary["feature_stats"].items(),
        key=lambda x: x[1]["count"],
        reverse=True,
    ):
        status = handled_features.get(feature, "❓ (unknown)")
        print(
            f"| {feature} | {status} | {data['count']:,} | {data['file_count']} |"
        )

feature_coverage_report()
